In [ ]:
%load_ext autoreload
%autoreload 2

# Embedding Quality Evaluation Pipeline

This notebook demonstrates the full `nlpie` workflow:
- **Preprocessing** — mean-center, L2-normalize, PCA whiten
- **Evaluation** — intrinsic, clustering, geometry, projection, retrieval metrics
- **Interpretation** — automatic explanations for each metric category
- **Visualization** — interactive Plotly dashboards, comparison charts
- **Export** — HTML, JSON, Markdown reports
- **Comparison** — side-by-side model evaluation with radar and delta charts

In [ ]:
import sys

import numpy as np

sys.path.insert(0, "..")
sys.path.insert(0, "../../python")

from nlpie import (
    DashboardBuilder,
    EmbeddingPreprocessor,
    HtmlExporter,
    JsonExporter,
    MarkdownExporter,
    compare_and_plot_delta,
    compare_and_plot_grouped_bar,
    compare_and_plot_radar,
    compute_hubness,
    cosine_similarity_matrix,
    explain_hubness,
    pairwise_cosine_stats,
    pearson_correlation,
    plot_comparison_radar,
    plot_embedding_scatter,
    plot_hubness_bar,
    plot_hubness_histogram,
    plot_projection_quality,
    plot_quality_report,
    plot_retrieval_metrics,
    plot_similarity_distribution,
    plot_similarity_heatmap,
    spearman_correlation,
)
from nlpie.metrics.quality import compare_models, evaluate_embedding_quality

In [ ]:
# Configure Plotly for Jupyter rendering
import plotly.io as pio

pio.renderers.default = "notebook_connected"

In [ ]:
# Configure Plotly for Jupyter rendering
import plotly.io as pio
pio.renderers.default = "notebook_connected"


## 1. Synthetic Data Generation

We create two synthetic embedding models:
- **Model A (Baseline)**: Noisy clusters with high overlap.
- **Model B (Finetuned)**: Tightly grouped clusters with clear separation.

In [ ]:
np.random.seed(42)
n_samples = 300
n_dims = 64
n_classes = 5

labels = np.random.randint(0, n_classes, n_samples)

centers_a = np.random.randn(n_classes, n_dims)
centers_b = np.random.randn(n_classes, n_dims) * 2.0

emb_a = centers_a[labels] + np.random.randn(n_samples, n_dims) * 1.5
emb_b = centers_b[labels] + np.random.randn(n_samples, n_dims) * 0.5

## 2. Preprocessing

We mean-center and L2-normalize. `nlpie`'s Rust backend makes this extremely fast.

In [ ]:
preprocessor = EmbeddingPreprocessor()

emb_a_centered, stats_a = preprocessor.mean_center(emb_a)
emb_a_norm = preprocessor.l2_normalize_rows(emb_a_centered)

emb_b_centered, stats_b = preprocessor.mean_center(emb_b)
emb_b_norm = preprocessor.l2_normalize_rows(emb_b_centered)

print(f"Model A column mean range: {min(stats_a.mean):.4f} to {max(stats_a.mean):.4f}")
print(f"Model B column mean range: {min(stats_b.mean):.4f} to {max(stats_b.mean):.4f}")

## 3. Correlation Analysis

We can compute pairwise correlations between any two vectors or embedding dimensions.

In [ ]:
pearson = pearson_correlation(emb_b_norm[0], emb_b_norm[1])
spearman = spearman_correlation(emb_b_norm[0], emb_b_norm[2])
print(f"Pearson r between point 0 and point 1: {pearson:.4f}")
print(f"Spearman ρ between point 0 and point 2: {spearman:.4f}")

## 4. Synthetic Projections & Retrieval Queries

To evaluate projection quality and retrieval, we need low-dimensional projections and retrieval queries.

In [ ]:
low_dim_a = np.array(emb_a_norm)[:, :2] + np.random.randn(n_samples, 2) * 0.1
low_dim_b = np.array(emb_b_norm)[:, :2]

n_queries = 10
retrieved_a = [np.random.choice(n_samples, 5, replace=False).tolist() for _ in range(n_queries)]
retrieved_b = [np.random.choice(n_samples, 5, replace=False).tolist() for _ in range(n_queries)]
relevant = [np.random.choice(n_samples, 2, replace=False).tolist() for _ in range(n_queries)]

## 5. Single Model Evaluation

`evaluate_embedding_quality` computes **all** metrics at once and returns both a report and an interpretation.

In [ ]:
report_b, interpretation_b = evaluate_embedding_quality(
    embeddings=emb_b_norm,
    labels=labels.tolist(),
    low_dim=low_dim_b.tolist(),
    projection_k_values=[5, 10],
    retrieved=retrieved_b,
    relevant=relevant,
    retrieval_k_values=[1, 3, 5],
    hubness_k=5,
    model_name="Model B (Finetuned)",
)

print(report_b)

### Interpretation Report

Each metric category includes a severity level, summary, and actionable recommendation.

In [ ]:
print(interpretation_b)

## 6. Visualizations

### 6a. Full Quality Dashboard

`plot_quality_report` renders all metric categories at once.

In [ ]:
# Full quality dashboard (KPI cards, charts, storytelling)
plot_quality_report(report_b).show()

### 6b. Individual Plot Functions

You can also call individual plot functions for fine-grained control.

In [ ]:
# Cosine similarity distribution (single Rust pass; no N×N matrix in Python)
vals, mean, std, _, _ = pairwise_cosine_stats(emb_b_norm)
plot_similarity_distribution(vals, mean, std).show()

In [ ]:
# Hubness histogram
counts, skewness = compute_hubness(emb_b_norm, k=5)
plot_hubness_histogram(counts, skewness, k=5).show()

In [ ]:
# Hubness bar chart (top hubs)
plot_hubness_bar(counts, top_n=20).show()

In [ ]:
# Projection quality curves
from nlpie import continuity, trustworthiness

k_vals = [1, 3, 5, 10, 15, 20]
t_vals = [trustworthiness(emb_b_norm, low_dim_b, k=k) for k in k_vals]
c_vals = [continuity(emb_b_norm, low_dim_b, k=k) for k in k_vals]
plot_projection_quality(k_vals, t_vals, c_vals).show()

In [ ]:
# Retrieval metrics curves
from nlpie import ndcg_at_k, precision_at_k, recall_at_k

k_vals = [1, 3, 5, 10]
rec_vals = [
    sum(recall_at_k(retrieved_b[i], relevant[i], k=k) for i in range(n_queries)) / n_queries
    for k in k_vals
]
prec_vals = [
    sum(precision_at_k(retrieved_b[i], relevant[i], k=k) for i in range(n_queries)) / n_queries
    for k in k_vals
]
ndcg_vals = [
    sum(ndcg_at_k(retrieved_b[i], relevant[i], k=k) for i in range(n_queries)) / n_queries
    for k in k_vals
]
plot_retrieval_metrics(k_vals, rec_vals, prec_vals, ndcg_vals).show()

In [ ]:
# Similarity heatmap (build the full matrix only where it is actually needed)
sim_mat = cosine_similarity_matrix(emb_b_norm)
plot_similarity_heatmap(sim_mat, labels=labels.tolist()).show()

In [ ]:
# Embedding scatter plot (2D projection)
plot_embedding_scatter(
    low_dim_b[:, 0], low_dim_b[:, 1], labels=labels.tolist(), title="Model B — 2D Projection"
).show()

## 7. Hubness Explanation

The `explain_hubness` function provides a structured interpretation of hubness pathology.

In [ ]:
explanation = explain_hubness(counts, skewness, k=5, top_n=3)
print(f"Severity: {explanation.severity}")
print(f"Summary: {explanation.summary}")
print(f"Skewness: {explanation.skewness:.3f}")
print(f"Top hubs: {[(h.index, h.count) for h in explanation.top_hubs]}")

## 8. DashboardBuilder

The `DashboardBuilder` provides a fluent API to compose a custom dashboard from selected sections.

In [ ]:
(
    DashboardBuilder(report_b)
    .add_hubness_histogram()
    .add_similarity_distribution()
    .add_projection_quality()
    .add_retrieval_metrics()
    .build()
    .show()
)

## 9. Multi-Model Comparison

`compare_models` evaluates multiple models on the same metrics.

In [ ]:
comparison_reports, comparison_interpretations = compare_models(
    models={"Model A (Baseline)": emb_a_norm, "Model B (Finetuned)": emb_b_norm},
    labels=labels.tolist(),
    low_dims={
        "Model A (Baseline)": low_dim_a.tolist(),
        "Model B (Finetuned)": low_dim_b.tolist(),
    },
    projection_k_values=[10],
    retrieved={
        "Model A (Baseline)": retrieved_a,
        "Model B (Finetuned)": retrieved_b,
    },
    relevant=relevant,
    retrieval_k_values=[5],
)

for report in comparison_reports:
    print(report)
    print()

for interp in comparison_interpretations:
    print(interp)
    print()

### Comparison Visualizations

Radar, grouped bar, and delta heatmap charts for side-by-side comparison.

In [ ]:
# Radar chart
fig = compare_and_plot_radar(
    {"Baseline": emb_a_norm, "Finetuned": emb_b_norm},
    labels=labels.tolist(),
    hubness_k=5,
)
fig.show()

In [ ]:
# Grouped bar chart
fig = compare_and_plot_grouped_bar(
    {"Baseline": emb_a_norm, "Finetuned": emb_b_norm},
    labels=labels.tolist(),
    hubness_k=5,
)
fig.show()

In [ ]:
# Delta heatmap (relative % change from baseline)
fig = compare_and_plot_delta(
    {"Baseline": emb_a_norm, "Finetuned": emb_b_norm},
    baseline="Baseline",
    labels=labels.tolist(),
    hubness_k=5,
)
fig.show()

### Manual Comparison Plots

You can also pass pre-computed reports to the plot functions directly.

In [ ]:
names = [r.model_name for r in comparison_reports]
plot_comparison_radar(
    names,
    ["ARI", "Silhouette", "Eff-Rank"],
    [
        [r.clustering.ari, r.clustering.silhouette, r.geometry.effective_rank / 64]
        for r in comparison_reports
    ],
).show()

## 10. Export Reports

`nlpie` can export the quality report to HTML, JSON, and Markdown.

In [ ]:
HtmlExporter().export(report_b, "quality_report.html")
JsonExporter().export(report_b, "quality_report.json")
MarkdownExporter().export(report_b, "quality_report.md")

print("Exported: quality_report.html, quality_report.json, quality_report.md")

In [ ]:
print(JsonExporter().to_string(report_b)[:500])